<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/DemoRAG%2CEmbeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install -q chromadb openai

In [21]:
# 1. What vectop database should we use? -> ChromaDB
# 2. What embedding model should we use? -> OpenAI embedding model

# Embedding vector, generated by AI model A cannot be directly compared to another embedding vector, generated by AI model B.

In [58]:
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from google.colab import userdata

openai_api_key = userdata.get('OPENAI_API_KEY')

chromadb_client = PersistentClient("/content/chromadb")
chromadb_collection = chromadb_client.get_or_create_collection(
      name = "books-v3",
      embedding_function = OpenAIEmbeddingFunction(api_key = openai_api_key, model_name = "text-embedding-3-large")
)

In [78]:
from pathlib import Path

def index_file(path_to_file:Path, chunk_size: int = 5, chunk_overlap_size: int = 2, batch_size: int = 100 ) -> None:
  with path_to_file.open() as file_content:
      file_lines = file_content.readlines()
      file_lines = [fl.strip() for fl in file_lines]
      file_lines = [fl for fl in file_lines if fl != '']

      chunks = []
      line_index = 0
      while line_index < len(file_lines):
          start = line_index
          end = line_index + chunk_size

          current = '\n'.join(file_lines[start:end])
          chunks.append({"text": current, "book": path_to_file.stem, "rel_line_start": start + 1, "rel_line_end":end})

          line_index = end - chunk_overlap_size

      chunk_index = 0
      batch_size = 100
      while chunk_index < len(chunks):
          start = chunk_index
          end = chunk_index + batch_size
          current_batch = chunks[start:end]

          # TODO: Id must be "book_name + line_start + line_end"
          # TODO: Add metadata
          # zapisvane v vektornata baza
          chromadb_collection.add(
              ids = [f"{x['book']}_{x['rel_line_start']}_{x['rel_line_end']}"for x in current_batch],
              documents = [x['text'] for x in current_batch],
              metadatas = [{"book": x['book'], "rel_line_start": x['rel_line_start'], 'rel_line_end': x['rel_line_end']} for x in current_batch]
          )

          #print(f"Successfully embedded documents in range [{start}, {end}]")
          chunk_index = end

In [76]:
index_file(Path("/content/Adventures of Sherlock Holmes.txt"))

Successfully embedded documents in range [0, 100]
Successfully embedded documents in range [100, 200]
Successfully embedded documents in range [200, 300]
Successfully embedded documents in range [300, 400]
Successfully embedded documents in range [400, 500]
Successfully embedded documents in range [500, 600]
Successfully embedded documents in range [600, 700]
Successfully embedded documents in range [700, 800]
Successfully embedded documents in range [800, 900]
Successfully embedded documents in range [900, 1000]
Successfully embedded documents in range [1000, 1100]
Successfully embedded documents in range [1100, 1200]
Successfully embedded documents in range [1200, 1300]
Successfully embedded documents in range [1300, 1400]
Successfully embedded documents in range [1400, 1500]
Successfully embedded documents in range [1500, 1600]
Successfully embedded documents in range [1600, 1700]
Successfully embedded documents in range [1700, 1800]
Successfully embedded documents in range [1800, 1

In [77]:
index_file(Path("/content/Alice's Adventures in Wonderland.txt"))

Successfully embedded documents in range [0, 100]
Successfully embedded documents in range [100, 200]
Successfully embedded documents in range [200, 300]
Successfully embedded documents in range [300, 400]
Successfully embedded documents in range [400, 500]
Successfully embedded documents in range [500, 600]
Successfully embedded documents in range [600, 700]
Successfully embedded documents in range [700, 800]
Successfully embedded documents in range [800, 900]
Successfully embedded documents in range [900, 1000]


In [40]:
#[start: end + 1 ]
#file_lines[1:5]

['This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this eBook or online']

In [79]:
results1 = chromadb_collection.query(
    query_texts = "color green",
    n_results = 5
)

In [80]:
results1

{'ids': [["Alice's Adventures in Wonderland_904_908",
   'Adventures of Sherlock Holmes_12382_12386',
   'Adventures of Sherlock Holmes_7465_7469',
   'Adventures of Sherlock Holmes_11428_11432',
   'Adventures of Sherlock Holmes_12742_12746']],
 'embeddings': None,
 'documents': [['“What _can_ all that green stuff be?” said Alice. “And where _have_ my\nshoulders got to? And oh, my poor hands, how is it I can’t see you?”\nShe was moving them about as she spoke, but no result seemed to follow,\nexcept a little shaking among the distant green leaves.\nAs there seemed to be no chance of getting her hands up to her head,',
   '\n\n\n\n\n\n\n\n\n',
   'a bright sun and a few fleecy clouds in the heavens. The trees and\n\nway-side hedges were just throwing out their first green shoots, and\n\nthe air was full of the pleasant smell of the moist earth. To me at\n\nleast there was a strange contrast between the sweet promise of the\n\nspring and this sinister quest upon which we were engaged. M